In [21]:
import tensorflow as tf
# pyrefly: ignore [missing-import]
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
# pyrefly: ignore [missing-import]
from tensorflow.keras.applications import MobileNetV2
# pyrefly: ignore [missing-import]
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Define constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_DIR = "data/dataset/train"
VAL_DIR = "data/dataset/val"

# 1. Load datasets automatically from directories
train_dataset = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"  # Class 0 or 1
)

val_dataset = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# IMPORTANT: confirm what index 0 / 1 actually mean before trusting any comment.
# class_names is alphabetical by subfolder name, e.g. ['cxr', 'non_cxr'] -> cxr=0, non_cxr=1
print("Class names (index order):", train_dataset.class_names)

# Optimize data loading performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# 2. Build the model using Transfer Learning
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False  # Freeze original weights

# Build model with preprocessing baked in, so train and inference can never drift apart
inputs = tf.keras.Input(shape=(224, 224, 3))
x = preprocess_input(inputs)              # scales [0,255] -> [-1,1] internally
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

# 3. Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 4. Train the model
print("Starting training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# 5. Save the trained model
model.save("chest_xray_detector.keras")
print("Model saved successfully as chest_xray_detector.keras")

# Print final val accuracy so a stuck-at-majority-class model is obvious immediately
print("Final val_accuracy:", history.history["val_accuracy"][-1])

Found 1773 files belonging to 2 classes.
Found 443 files belonging to 2 classes.
Class names (index order): ['non_x-ray', 'x-ray']


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Starting training...
Epoch 1/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 13s 184ms/step - accuracy: 0.9380 - loss: 0.1683 - val_accuracy: 0.9910 - val_loss: 0.0323
Epoch 2/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 147ms/step - accuracy: 0.9938 - loss: 0.0295 - val_accuracy: 0.9910 - val_loss: 0.0211
Epoch 3/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 139ms/step - accuracy: 0.9966 - loss: 0.0198 - val_accuracy: 0.9932 - val_loss: 0.0155
Epoch 4/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 137ms/step - accuracy: 0.9977 - loss: 0.0168 - val_accuracy: 0.9955 - val_loss: 0.0125
Epoch 5/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 137ms/step - accuracy: 0.9983 - loss: 0.0148 - val_accuracy: 0.9977 - val_loss: 0.0097
Model saved successfully as chest_xray_detector.keras
Final val_accuracy: 0.9977426528930664


In [3]:
# pyrefly: ignore [missing-import]
import numpy as np
from tensorflow.keras.models import load_model
# pyrefly: ignore [missing-import]
from tensorflow.keras.preprocessing import image
from utils import predict_if_cxr

# Load your trained model
model = load_model("models/chest_xray_detector.keras")

# Example usage:
predict_if_cxr("/Users/wess/Desktop/computer vision/Pneumonia Classifcation 🫁/data/dataset/train/x-ray/1-s2.0-S0929664620300449-gr2_lrg-c.jpg", model)


Result: This IS a Chest X-Ray (99.96% confidence)


True